# Module 1.5 — The Complete Physics Research Assistant

**Where we are — the SA Part climax.** Over the last four notebooks we built five independent pieces:

| Module | What we added | Notebook |
|---|---|---|
| 1.0 | A stateless `chat(messages, T) -> str` primitive | `01_naked_llm` |
| 1.1 | The ReAct loop + `calculator` + `unit_converter` | `02_react_agent` |
| 1.2 | Retrieval from an arXiv corpus (`search_papers` via ChromaDB) | `03_rag_agent` |
| 1.3 | External tools over MCP (`search_arxiv`, `run_ising_simulation`) | `04_mcp_tools` |
| 1.4 | Reflection + Reflexion (Critic + JSON-file memory) | `05_reflection_agent` |

This notebook adds **no new capabilities.** It does exactly one thing: slot all five into a single agent and point it at one real end-to-end research task.

**The architecture we're assembling:**

```
┌─────────────────────────────────────────────────────────────┐
│            Physics Research Assistant (SA Part)               │
│                                                             │
│   ┌─────────────────────────────────────────────────────┐   │
│   │         smolagents CodeAgent — ReAct loop           │   │
│   │  Thought → write Python → observe → … → final_answer│   │
│   └────────────────────────┬────────────────────────────┘   │
│                            │                                │
│      ┌─────────────────────┼────────────────────────┐       │
│      │                     │                        │       │
│  ┌───▼──────┐   ┌──────────▼─────────┐   ┌──────────▼────┐  │
│  │  local   │   │        RAG         │   │     MCP       │  │
│  │  tools   │   │                    │   │  (physics     │  │
│  │          │   │  search_papers     │   │   tools       │  │
│  │ calc-    │   │  ⇢ ChromaDB        │   │   server)     │  │
│  │ ulator   │   │  ⇢ sentence-       │   │               │  │
│  │ unit_    │   │    transformers    │   │  search_arxiv │  │
│  │ conv-    │   │                    │   │  run_ising_   │  │
│  │ erter    │   │                    │   │   simulation  │  │
│  └──────────┘   └────────────────────┘   └───────────────┘  │
│                                                             │
│   ┌─────────────────────────────────────────────────────┐   │
│   │   Reflection (within-task) + Reflexion (across)     │   │
│   │   Critic — one LLM call, JSON verdict                │   │
│   │   ReflexionMemory — data/reflexion_memory.json      │   │
│   └─────────────────────────────────────────────────────┘   │
│                                                             │
│   LLM backend: Ollama / qwen2.5:7b (same model throughout)  │
└─────────────────────────────────────────────────────────────┘
```

No new loops, no new memory, no new tools — only assembly. That's the point of this module: a modern agent is **five things glued together**, and you've now built every one of them.

**What we'll do.** One end-to-end research task, exactly as specified in the course outline. Then we look honestly at what the single-agent paradigm handles well and where it strains — which is the motivation for the MAS Part (specialised multi-agent teams).


## 1. Setup — MCP connection, ChromaDB, encoder

Three pieces of infrastructure to bring up, all of which we've used before:

1. **MCP server** — a child process running `mcp_server.physics_tools_server`. `ToolCollection.from_mcp` is a context manager; we enter it manually here so the connection stays open across cells, and we exit it explicitly in §6.
2. **ChromaDB** — the persistent vector index built in Module 1.2. We *open* it (no re-indexing), so this cell is cheap as long as the index already exists on disk.
3. **Sentence-transformers encoder** — `all-MiniLM-L6-v2`, the same model Module 1.2 used to embed the corpus. We need it to embed the agent's query at search time.

If `chroma_db/` doesn't exist (because you skipped Module 1.2), this cell will fail with a clear error and the fix is to run `03_rag_agent.ipynb` once first.

One extra bit of plumbing lives at the end of the code cell below: we wrap the MCP `run_ising_simulation` tool in a thin local adapter that automatically `json.loads`-es its output, so the agent sees a native Python dict instead of a JSON-encoded string. This is the kind of adapter a production MCP client would add anyway — it keeps the wire format clean (JSON) while sparing the model from remembering to parse on every call. Without it, a 7B-class agent reliably forgets the parse step and locks into a `subscript-a-string` retry loop.

In [1]:
import os
import sys
import warnings
from pathlib import Path

# Silence the known-harmless LiteLLM/Pydantic v2 warning, as in 1.3 and 1.4.
warnings.filterwarnings("ignore", category=UserWarning, module="pydantic")

from mcp import StdioServerParameters
from smolagents import ToolCollection, CodeAgent, LiteLLMModel

# ---------------------------------------------------------------------------
# 1a. MCP connection -- same pattern as 04 and 05
# ---------------------------------------------------------------------------
SERVER_PARAMS = StdioServerParameters(
    command=sys.executable,
    args=["-m", "mcp_server.physics_tools_server"],
    env={**os.environ, "PYTHONPATH": ".."},
    cwd="..",
)
_mcp_cm = ToolCollection.from_mcp(SERVER_PARAMS)
tc = _mcp_cm.__enter__()
print(f"MCP tools discovered: {[t.name for t in tc.tools]}")

# ---------------------------------------------------------------------------
# 1b. ChromaDB -- *open* the persistent collection built in Module 1.2
# ---------------------------------------------------------------------------
import chromadb

CHROMA_DIR      = Path("../chroma_db")
COLLECTION_NAME = "ising_papers"

if not CHROMA_DIR.exists():
    raise RuntimeError(
        f"No ChromaDB at {CHROMA_DIR.resolve()}. "
        "Run `notebooks/03_rag_agent.ipynb` once to build the index."
    )

client = chromadb.PersistentClient(path=str(CHROMA_DIR))
try:
    collection = client.get_collection(COLLECTION_NAME)
except Exception as exc:
    raise RuntimeError(
        f"Collection {COLLECTION_NAME!r} is missing from {CHROMA_DIR}. "
        "Re-run Module 1.2 to rebuild it."
    ) from exc
print(f"ChromaDB: opened collection '{COLLECTION_NAME}' with {collection.count()} chunks")

# ---------------------------------------------------------------------------
# 1c. Sentence-transformers encoder (same model as Module 1.2)
# ---------------------------------------------------------------------------
from sentence_transformers import SentenceTransformer
encoder = SentenceTransformer("all-MiniLM-L6-v2")
print(f"Encoder ready: all-MiniLM-L6-v2 (dim={encoder.get_sentence_embedding_dimension()})")

# ---------------------------------------------------------------------------
# 1d. Local wrapper for run_ising_simulation -- auto-parses the MCP JSON
# ---------------------------------------------------------------------------
# MCP tools return *strings* (JSON-encoded on the wire via mcpadapt). A 7B
# agent, after seeing `print(result)` produce something that *looks* like a
# dict, reliably forgets to call `json.loads` and then locks into a
# `subscript-a-string-with-a-string-index` retry loop. We short-circuit that
# whole failure class by wrapping the MCP tool in a local `@tool` adapter
# that does the parse for the agent. From the Actor's point of view,
# `run_ising_simulation(...)` just returns a native Python dict.
#
# We do NOT wrap `search_arxiv` the same way -- its output is smaller, its
# parse-on-use pattern is part of the Module 1.3 MCP lesson, and keeping
# one tool unwrapped preserves that teaching point.
from smolagents import tool
import json as _json_wrap

_mcp_run_ising = {t.name: t for t in tc.tools}["run_ising_simulation"]


@tool
def run_ising_simulation(
    lattice_size: int,
    temperature: float,
    num_steps: int,
    algorithm: str = "wolff",
    thermalization_steps: int | None = None,
    seed: int | None = None,
) -> dict:
    """Run a 2D Ising Monte Carlo simulation; return observables as a dict.

    Units: J = 1, k_B = 1. Onsager's exact T_c = 2 / ln(1 + sqrt(2)) ~= 2.269.

    Args:
        lattice_size: linear size L of the L x L lattice (>= 2). Typical: 16, 32, 64.
        temperature: temperature T in units of J/k_B (must be > 0).
        num_steps: number of Monte Carlo measurement steps (after thermalisation).
        algorithm: 'metropolis' or 'wolff' (default 'wolff', preferred near T_c).
        thermalization_steps: optional; defaults to num_steps // 5.
        seed: optional RNG seed for reproducibility.
    """
    kwargs: dict = {
        "lattice_size": lattice_size,
        "temperature":  temperature,
        "num_steps":    num_steps,
        "algorithm":    algorithm,
    }
    if thermalization_steps is not None:
        kwargs["thermalization_steps"] = thermalization_steps
    if seed is not None:
        kwargs["seed"] = seed
    raw = _mcp_run_ising(**kwargs)
    return _json_wrap.loads(raw) if isinstance(raw, str) else raw


# Keep every *other* MCP tool (i.e. search_arxiv) for the agent assembly in §3.
mcp_tools_for_agent = [t for t in tc.tools if t.name != "run_ising_simulation"]
print(
    "run_ising_simulation wrapped (MCP -> dict).  "
    f"Other MCP tools passed through: {[t.name for t in mcp_tools_for_agent]}"
)


MCP tools discovered: ['search_arxiv', 'run_ising_simulation']
ChromaDB: opened collection 'ising_papers' with 97 chunks
Encoder ready: all-MiniLM-L6-v2 (dim=384)
run_ising_simulation wrapped (MCP -> dict).  Other MCP tools passed through: ['search_arxiv']


## 2. The three local tools

`calculator_tool` and `unit_converter_tool` from Module 1.1, `search_papers` from Module 1.2. The `@tool` decorator from `smolagents` inspects the *source* of each function, which is why they need to be defined in this notebook rather than imported from a sibling notebook's kernel.

> **For the course they live inline; outside the course they'd sit in a shared `tools/` module** — same comment notebook 03 made when it first needed to reuse the calculator. The duplication is a feature of the teaching format, not a design recommendation.

One small diff from 1.1: the calculator's allowed-names whitelist gains `sinh`, `cosh`, `tanh`. Onsager's magnetization formula $|M|(T) = (1 - \sinh(2/T)^{-4})^{1/8}$ uses `sinh`, and we'd like the agent to be able to evaluate it via the tool rather than by hand. Everything else in these three tools is identical to where it was introduced.


In [2]:
import ast
import math
import operator as op
from smolagents import tool
from typing import Any

# ---------------------------------------------------------------------------
# 2a. calculator_tool (from Module 1.1, with hyperbolic-trig extension).
# Onsager's |M|(T) = (1 - sinh(2/T)^(-4))^(1/8) needs sinh, so we extend the
# allowed-names whitelist with sinh/cosh/tanh. That is the only diff from 1.1.
# ---------------------------------------------------------------------------
_ALLOWED_BINOPS = {
    ast.Add: op.add, ast.Sub: op.sub, ast.Mult: op.mul,
    ast.Div: op.truediv, ast.Pow: op.pow, ast.Mod: op.mod,
    ast.FloorDiv: op.floordiv,
}
_ALLOWED_UNARYOPS = {ast.UAdd: op.pos, ast.USub: op.neg}
_ALLOWED_NAMES: dict[str, Any] = {
    "pi": math.pi, "e": math.e,
    "sqrt": math.sqrt, "log": math.log, "log10": math.log10,
    "exp": math.exp, "sin": math.sin, "cos": math.cos, "tan": math.tan,
    "asin": math.asin, "acos": math.acos, "atan": math.atan,
    "sinh": math.sinh, "cosh": math.cosh, "tanh": math.tanh,  # extension for Onsager
    "abs": abs,
}

def _safe_eval(node: ast.AST) -> float:
    if isinstance(node, ast.Expression):
        return _safe_eval(node.body)
    if isinstance(node, ast.Constant) and isinstance(node.value, (int, float)):
        return float(node.value)
    if isinstance(node, ast.Name) and node.id in _ALLOWED_NAMES:
        v = _ALLOWED_NAMES[node.id]
        return v if callable(v) else float(v)
    if isinstance(node, ast.BinOp) and type(node.op) in _ALLOWED_BINOPS:
        return _ALLOWED_BINOPS[type(node.op)](_safe_eval(node.left), _safe_eval(node.right))
    if isinstance(node, ast.UnaryOp) and type(node.op) in _ALLOWED_UNARYOPS:
        return _ALLOWED_UNARYOPS[type(node.op)](_safe_eval(node.operand))
    if isinstance(node, ast.Call) and isinstance(node.func, ast.Name):
        fn = _ALLOWED_NAMES.get(node.func.id)
        if callable(fn) and not node.keywords:
            return fn(*[_safe_eval(a) for a in node.args])
    raise ValueError(f"disallowed expression: {ast.dump(node)}")


@tool
def calculator_tool(expression: str) -> float:
    """Evaluate a math expression.

    Supports +, -, *, /, **, sqrt, log, log10, exp, sin, cos, tan, sinh, cosh,
    tanh, abs, pi, e.

    Args:
        expression: a math expression, e.g. "2/log(1+sqrt(2))" for T_c.
    """
    return float(_safe_eval(ast.parse(expression, mode="eval")))


# ---------------------------------------------------------------------------
# 2b. unit_converter_tool (verbatim from Module 1.1)
# ---------------------------------------------------------------------------
_TO_SI: dict[str, tuple[str, float]] = {
    "J":   ("energy", 1.0),
    "eV":  ("energy", 1.602176634e-19),
    "keV": ("energy", 1.602176634e-16),
    "MeV": ("energy", 1.602176634e-13),
    "GeV": ("energy", 1.602176634e-10),
    "erg": ("energy", 1.0e-7),
    "m":   ("length", 1.0),
    "km":  ("length", 1.0e3),
    "cm":  ("length", 1.0e-2),
    "mm":  ("length", 1.0e-3),
    "nm":  ("length", 1.0e-9),
    "A":   ("length", 1.0e-10),
    "kg":   ("mass", 1.0),
    "g":    ("mass", 1.0e-3),
    "Msun": ("mass", 1.98892e30),
}
_K_B_JPK = 1.380649e-23

def _unit_convert(value: float, from_unit: str, to_unit: str) -> float:
    if from_unit == "K" and to_unit == "K":
        return float(value)
    if from_unit == "K" and to_unit in _TO_SI and _TO_SI[to_unit][0] == "energy":
        joules = float(value) * _K_B_JPK
        return joules / _TO_SI[to_unit][1]
    if to_unit == "K" and from_unit in _TO_SI and _TO_SI[from_unit][0] == "energy":
        joules = float(value) * _TO_SI[from_unit][1]
        return joules / _K_B_JPK
    if from_unit not in _TO_SI or to_unit not in _TO_SI:
        raise ValueError(f"unknown unit: {from_unit!r} or {to_unit!r}")
    dim_from, f_si = _TO_SI[from_unit]
    dim_to,   t_si = _TO_SI[to_unit]
    if dim_from != dim_to:
        raise ValueError(f"dimension mismatch: {from_unit} vs {to_unit}")
    return float(value) * f_si / t_si


@tool
def unit_converter_tool(value: float, from_unit: str, to_unit: str) -> float:
    """Convert a value between units.

    Supported dimensions: energy (J, eV, keV, MeV, GeV, erg),
    length (m, km, cm, mm, nm, A), mass (kg, g, Msun), and
    temperature <-> energy via k_B (K <-> J, eV, ...).

    Args:
        value: numeric value to convert.
        from_unit: source unit, e.g. "eV".
        to_unit: target unit, e.g. "J".
    """
    return _unit_convert(value, from_unit, to_unit)


# ---------------------------------------------------------------------------
# 2c. search_papers (verbatim from Module 1.2)
# ---------------------------------------------------------------------------
def _retrieve(query: str, top_k: int = 3) -> list[dict]:
    q_emb = encoder.encode([query], normalize_embeddings=False).tolist()
    res = collection.query(query_embeddings=q_emb, n_results=top_k)
    hits: list[dict] = []
    for i in range(len(res["ids"][0])):
        hits.append({
            "doc_id":   res["ids"][0][i],
            "distance": float(res["distances"][0][i]),
            "text":     res["documents"][0][i],
            "meta":     res["metadatas"][0][i],
        })
    return hits


@tool
def search_papers(query: str, top_k: int = 3) -> str:
    """Search the local arXiv corpus (Ising-model papers) for passages relevant to a query.

    Returns a newline-separated list of snippets with citations. Use this tool when
    the question is about what specific papers say: methods, results, claims,
    numerical values, historical attribution.

    Args:
        query: a natural-language question or search phrase.
        top_k: how many passages to return (1-5). Default 3.
    """
    hits = _retrieve(query, top_k=max(1, min(top_k, 5)))
    lines = []
    for i, h in enumerate(hits):
        m = h["meta"]
        snippet = h["text"][:500].replace("\n", " ")
        lines.append(
            f"[{i+1}] {m['title']} (arXiv:{m['arxiv_id']}, p.{m['page_start']}): {snippet}..."
        )
    return "\n\n".join(lines) if lines else "(no relevant passages found)"


print("Local tools ready:",
      [calculator_tool.name, unit_converter_tool.name, search_papers.name])


Local tools ready: ['calculator_tool', 'unit_converter_tool', 'search_papers']


## 3. Assembly — one agent, five tools, one reflection wrapper

We build a single `CodeAgent` with all five tools exposed: three local (§2) and two over MCP (§1). The agent sees them as a flat list — it does not know or care where each tool came from. That's precisely the *uniform tool-use interface* that makes MCP valuable.

Two knobs that matter for this module specifically:

- `max_steps=15` — the task in §4 can involve 3+ MCP simulation calls, a RAG query, and one or two arithmetic checks. The ReAct chain is deeper than in any earlier module, and Reflection revisions add yet more steps. Fifteen gives the Actor comfortable headroom.
- `additional_authorized_imports=["math", "json", "numpy"]` — needed so the agent can `json.loads(...)` MCP tool returns (see Module 1.3 §7) and do any numerical comparison inline.

We also bring in `critique`, `run_with_reflection`, and `ReflexionMemory` from Module 1.4 (re-defined inline — same "would live in a `tools/` module outside the course" caveat). The Critic prompt is copied over verbatim from the 1.4 refinement.


In [3]:
import json as _json
import json           # full module for use inside the class/functions
import re
from litellm import completion

MODEL = "qwen2.5:7b"

# ---------------------------------------------------------------------------
# 3a. Actor -- one CodeAgent holding all 5 tools
# ---------------------------------------------------------------------------
actor_model = LiteLLMModel(
    model_id=f"ollama_chat/{MODEL}",
    api_base="http://localhost:11434",
    api_key="ollama",
    temperature=0.0,
)

agent = CodeAgent(
    tools=[calculator_tool, unit_converter_tool, search_papers,
           run_ising_simulation,          # local wrapper from cell 1d (returns a dict)
           *mcp_tools_for_agent],         # everything else from MCP (search_arxiv)
    model=actor_model,
    additional_authorized_imports=["math", "json", "numpy"],
    max_steps=15,
)

print("Actor ready. Tools on this agent:")
for name in agent.tools.keys():
    print(f"  - {name}")


# ---------------------------------------------------------------------------
# 3b. Critic -- sharpened further on top of Module 1.4's version.
# Rule 3 now explicitly demands multi-part answers for multi-part tasks
# (first run of this module produced a bare-float answer on a three-sim
# task, and the Critic waved it through with a mild "needs more detail"
# remark -- this is the fix).
# ---------------------------------------------------------------------------
CRITIC_MODEL = f"ollama_chat/{MODEL}"

CRITIC_SYSTEM = """You are a senior physicist reviewing a junior colleague's answer. Be strict but fair.

Rules you MUST follow, in order:

 1. VERIFY before accusing.
    - If the junior lists numbers and claims a maximum / minimum / peak, check it against the numbers they show. Do NOT invent arithmetic errors that are not there. If the listed value is indeed the largest/smallest of the shown values, accept that part of the answer.
    - If the junior does NOT show the underlying numbers, you may ask them to include the raw values -- but do NOT assert that their stated value is wrong when you have no data to check it against.

 2. For any claim about the 2D Ising critical temperature T_c (units J=1, k_B=1), compare to Onsager's exact value
        T_c = 2 / ln(1 + sqrt(2))  ~=  2.269.
    If the estimate differs by more than ~2% AND it was taken from a small lattice (L <= 16), REJECT and ask the junior to either rerun on a larger lattice (L >= 24) or explicitly acknowledge the finite-size shift in the answer.

 3. Reject raw-JSON dumps, hand-waving ("roughly around 2-ish"), or answers that do not address the question that was asked. Specifically:
    - If the task asks the junior to COMPARE simulation results to theoretical predictions, the answer must state both values and the percentage difference.
    - If the task asks for N specific measurements or comparisons (e.g. "at three temperatures", "for each of three lattice sizes"), the answer MUST show all N. A bare number, a one-sentence qualitative remark, or an answer covering only part of the task is always a REJECT -- regardless of whether the fragment it does cover happens to be correct. Your feedback should name the missing pieces explicitly (e.g. "missing magnetization and energy at T=2.0 and T=2.5").

 4. Otherwise, if the physics is sound and the numbers are in reasonable ranges, ACCEPT. Do not invent fresh objections on revisions just to keep rejecting.

Respond with a JSON object and nothing else, of the form:

  {"verdict": "accept" | "revise",
   "feedback": "<1-2 sentences explaining the verdict, citing specific numbers from the answer>",
   "lesson":   "<if verdict==revise, ONE short imperative sentence (<=15 words) the junior should remember next time; else empty string>"}
"""


def critique(task: str, answer: str, *, model: str = CRITIC_MODEL) -> dict:
    """One LLM call; returns {"verdict", "feedback", "lesson"}."""
    user = (
        f"Task given to the junior:\n{task}\n\n"
        f"Junior's answer:\n{answer}\n\n"
        "Return only the JSON object specified in the system prompt."
    )
    resp = completion(
        model=model,
        api_base="http://localhost:11434",
        api_key="ollama",
        messages=[
            {"role": "system", "content": CRITIC_SYSTEM},
            {"role": "user",   "content": user},
        ],
        temperature=0.0,
    )
    raw = resp["choices"][0]["message"]["content"]
    m = re.search(r"\{.*\}", raw, flags=re.DOTALL)
    if m is None:
        return {"verdict": "accept", "feedback": f"Critic non-JSON: {raw[:120]}...", "lesson": ""}
    try:
        parsed = _json.loads(m.group(0))
    except _json.JSONDecodeError:
        return {"verdict": "accept", "feedback": f"Critic JSON fail: {raw[:120]}...", "lesson": ""}
    parsed.setdefault("verdict", "accept")
    parsed.setdefault("feedback", "")
    parsed.setdefault("lesson", "")
    return parsed


# ---------------------------------------------------------------------------
# 3c. Reflection loop -- verbatim from Module 1.4
# ---------------------------------------------------------------------------
def run_with_reflection(
    agent,
    task: str,
    *,
    max_retries: int = 2,
    memory_lessons: list[str] | None = None,
    verbose: bool = True,
) -> dict:
    lessons_this_task: list[str] = []
    transcript: list[dict] = []

    prompt = task
    if memory_lessons:
        bullets = "\n".join(f"- {l}" for l in memory_lessons)
        prompt = (
            "Lessons from prior tasks (apply them if relevant):\n"
            f"{bullets}\n\n"
            f"Task:\n{task}"
        )

    for attempt in range(max_retries + 1):
        if verbose:
            print(f"\n=== Attempt {attempt + 1} ===")
        agent.memory.reset()
        answer = agent.run(prompt)
        if verbose:
            print("\n--- Actor answer ---")
            print(str(answer)[:500] + ("..." if len(str(answer)) > 500 else ""))

        verdict = critique(task, str(answer))
        transcript.append({"answer": str(answer), "critique": verdict})
        if verbose:
            print(f"\n--- Critic: {verdict['verdict']} ---")
            print(verdict["feedback"])

        if verdict["verdict"] == "accept":
            return {"final_answer": str(answer),
                    "accepted_on_attempt": attempt + 1,
                    "transcript": transcript,
                    "lessons": lessons_this_task}

        if verdict["lesson"]:
            lessons_this_task.append(verdict["lesson"])

        prompt = (
            f"{task}\n\n"
            f"Your previous answer was rejected by the reviewer with this feedback:\n"
            f"  {verdict['feedback']}\n\n"
            "Revise your answer to address that feedback. Remember: the reviewer "
            "reads ONLY the string you pass to `final_answer(...)`. Stdout prints "
            "are invisible to them. Make `final_answer` self-contained -- restate "
            "all numerical results inside that string."
        )

    return {"final_answer": str(answer),
            "accepted_on_attempt": None,
            "transcript": transcript,
            "lessons": lessons_this_task}


# ---------------------------------------------------------------------------
# 3d. Reflexion memory -- verbatim from Module 1.4
# ---------------------------------------------------------------------------
MEMORY_PATH = Path("../data/reflexion_memory.json")

class ReflexionMemory:
    def __init__(self, path: Path = MEMORY_PATH):
        self.path = Path(path)
        self.path.parent.mkdir(parents=True, exist_ok=True)
        if not self.path.exists():
            self.path.write_text("[]")

    def lessons(self) -> list[str]:
        return json.loads(self.path.read_text())

    def add(self, lesson: str) -> None:
        if not lesson:
            return
        cur = self.lessons()
        if lesson in cur:
            return
        cur.append(lesson)
        self.path.write_text(json.dumps(cur, indent=2))

    def clear(self) -> None:
        self.path.write_text("[]")


memory = ReflexionMemory()
memory.clear()  # start the Day 1 climax with a clean slate
print(f"\nReflexion memory reset: {memory.path.resolve()}")


Actor ready. Tools on this agent:
  - calculator_tool
  - unit_converter_tool
  - search_papers
  - run_ising_simulation
  - search_arxiv
  - final_answer

Reflexion memory reset: /Users/Yak52/coding-lab/ising-agents-course/data/reflexion_memory.json


## 4. The end-to-end task

This is the task specified in the course outline, verbatim:

> *"I'm starting a project on the 2D Ising model. Search our paper corpus for recommended simulation parameters, then run a quick simulation at three temperatures (below, at, and above $T_c$) on a 32×32 lattice. Report the magnetization and energy at each temperature, and check whether your results are consistent with the theoretical predictions."*

**What a good run looks like, step by step:**

1. `search_papers("recommended simulation parameters for 2D Ising Monte Carlo")` — finds one or two hits in the corpus (typical: Wolff cluster updates, ≥5000 steps, thermalisation guidance).
2. Pick three temperatures bracketing $T_c \approx 2.269$: e.g. $T = 2.0, 2.27, 2.5$.
3. Three calls to `run_ising_simulation(lattice_size=32, T=<T>, num_steps=5000, algorithm="wolff")`. Each returns a JSON-encoded string — the agent must `json.loads(...)` it (Module 1.3 §7 pattern).
4. Read `magnetization_mean` and `energy_mean` from each result. Optionally use `calculator_tool` to compute Onsager's $|M|(T) = (1 - \sinh(2/T)^{-4})^{1/8}$ for the $T < T_c$ point.
5. Summarise: magnetization high below $T_c$, near-zero above, noisy near the transition; energy monotonically increasing with $T$. Compare to theory where available.

**A few practical hints in the prompt below.** Reflection-era learnings from Module 1.4 are already baked in: *"`final_answer` must be self-contained"*, *"the reviewer cannot see stdout"*, *"`run_ising_simulation` returns a JSON string — `json.loads` it before indexing"*. Those hints are not new science; they are the Actor/Critic contract we arrived at in 1.4, carried forward.

> **Wall-clock expectation.** One simulation of a 32×32 lattice for 5000 Wolff steps is a few seconds. With three simulations plus one RAG call, one or two Critic cycles, and qwen2.5:7b's ReAct overhead, a full run typically takes **2–5 minutes**. Watch the streaming output to see the agent think.


In [4]:
# The headline task -- copied verbatim from the course outline.
TASK_HEADLINE = (
    "I'm starting a project on the 2D Ising model. Search our paper corpus for "
    "recommended simulation parameters, then run a quick simulation at three "
    "temperatures (below, at, and above T_c) on a 32x32 lattice. Report the "
    "magnetization and energy at each temperature, and check whether your "
    "results are consistent with the theoretical predictions."
)

# Tack on the Actor/Critic contract hints we earned in Module 1.4 plus the
# additional ones we learned the hard way on the first run of this very
# module (see §6 for the autopsy). The headline is what a student-user would
# actually type; everything below the blank line is agent-engineering glue --
# the same kind of guardrail a real deployment would hide in a system prompt.
TASK = TASK_HEADLINE + "\n\n" + (
    "Practical notes:\n"
    " - T_c for the 2D Ising model (J=1, k_B=1) is Onsager's exact value "
    "   2 / ln(1 + sqrt(2)) ~= 2.269. Pick three temperatures bracketing this, "
    "   e.g. T = 2.0, 2.27, 2.5.\n"
    " - The tools `search_papers`, `calculator_tool`, `unit_converter_tool`, "
    "   `run_ising_simulation`, and `search_arxiv` are already in scope as "
    "   Python functions. CALL THEM DIRECTLY, e.g. "
    "   `result = run_ising_simulation(lattice_size=32, temperature=2.0, num_steps=5000, algorithm='wolff')`. "
    "   The temperature kwarg is `temperature=`, NOT `T=` -- using `T=` raises "
    "   a pydantic validation error. "
    "   Do NOT write `import run_ising_simulation` -- tools are not Python "
    "   modules and the sandbox will reject that import.\n"
    " - `final_answer` is also a tool. Call it with `final_answer(\"your answer string\")` "
    "   to return your final answer. It takes EXACTLY ONE string argument -- do "
    "   NOT call `final_answer(a, b)` with multiple values, and do NOT write "
    "   `final_answer = \"...\"` (that assignment will fail with 'Cannot assign "
    "   to name final_answer').\n"
    " - `run_ising_simulation(...)` returns a native Python dict; index it "
    "   directly with `result['magnetization_mean']` or `result['energy_mean']` "
    "   (no `json.loads` needed -- a local adapter handles that for you).\n"
    " - For T < T_c on an infinite lattice, Onsager gives "
    "   |M|(T) = (1 - sinh(2/T)^(-4))^(1/8). You can evaluate this with "
    "   `calculator_tool`.\n"
    " - Your `final_answer(...)` must be a single self-contained plain-English "
    "   string that restates (T, magnetization, energy) for ALL THREE "
    "   temperatures and says explicitly how each result compares to theory. "
    "   The reviewer reads ONLY the string you pass to `final_answer`; "
    "   stdout prints are invisible to them. A bare number or a one-sentence "
    "   qualitative remark is NOT an acceptable answer for this three-part "
    "   task."
)

print(TASK)


I'm starting a project on the 2D Ising model. Search our paper corpus for recommended simulation parameters, then run a quick simulation at three temperatures (below, at, and above T_c) on a 32x32 lattice. Report the magnetization and energy at each temperature, and check whether your results are consistent with the theoretical predictions.

Practical notes:
 - T_c for the 2D Ising model (J=1, k_B=1) is Onsager's exact value    2 / ln(1 + sqrt(2)) ~= 2.269. Pick three temperatures bracketing this,    e.g. T = 2.0, 2.27, 2.5.
 - The tools `search_papers`, `calculator_tool`, `unit_converter_tool`,    `run_ising_simulation`, and `search_arxiv` are already in scope as    Python functions. CALL THEM DIRECTLY, e.g.    `result = run_ising_simulation(lattice_size=32, temperature=2.0, num_steps=5000, algorithm='wolff')`.    The temperature kwarg is `temperature=`, NOT `T=` -- using `T=` raises    a pydantic validation error.    Do NOT write `import run_ising_simulation` -- tools are not Python 

## 5. The demo run

Everything the course has built, one cell, one task. Expect several minutes and a dense trace — the Actor will think, call `search_papers`, call `run_ising_simulation` three times, do arithmetic, and attempt a final answer. The Critic will then either accept or send it back with feedback.


In [5]:
result = run_with_reflection(
    agent,
    TASK,
    max_retries=2,
    memory_lessons=memory.lessons(),  # empty: fresh session
)

for l in result["lessons"]:
    memory.add(l)

print("\n================ END-TO-END SUMMARY ================")
print("Accepted on attempt :", result["accepted_on_attempt"])
print("Lessons distilled   :", result["lessons"])
print("\nFinal answer:")
ans = str(result["final_answer"])
print(ans[:1200] + ("..." if len(ans) > 1200 else ""))



=== Attempt 1 ===


╭──────────────────────────────────────────────────── New run ────────────────────────────────────────────────────╮
│                                                                                                                 │
│ I'm starting a project on the 2D Ising model. Search our paper corpus for recommended simulation parameters,    │
│ then run a quick simulation at three temperatures (below, at, and above T_c) on a 32x32 lattice. Report the     │
│ magnetization and energy at each temperature, and check whether your results are consistent with the            │
│ theoretical predictions.                                                                                        │
│                                                                                                                 │
│ Practical notes:                                                                                                │
│  - T_c for the 2D Ising model (J=1, k_B=1) is Onsager's exact value    2 / ln(1 + sqrt(2)) ~= 2.269. Pick three │
│ temperatures bracketing this,    e.g. T = 2.0, 2.27, 2.5.                                                       │
│  - The tools `search_papers`, `calculator_tool`, `unit_converter_tool`,    `run_ising_simulation`, and          │
│ `search_arxiv` are already in scope as    Python functions. CALL THEM DIRECTLY, e.g.    `result =               │
│ run_ising_simulation(lattice_size=32, temperature=2.0, num_steps=5000, algorithm='wolff')`.    The temperature  │
│ kwarg is `temperature=`, NOT `T=` -- using `T=` raises    a pydantic validation error.    Do NOT write `import  │
│ run_ising_simulation` -- tools are not Python    modules and the sandbox will reject that import.               │
│  - `final_answer` is also a tool. Call it with `final_answer("your answer string")`    to return your final     │
│ answer. It takes EXACTLY ONE string argument -- do    NOT call `final_answer(a, b)` with multiple values, and   │
│ do NOT write    `final_answer = "..."` (that assignment will fail with 'Cannot assign    to name                │
│ final_answer').                                                                                                 │
│  - `run_ising_simulation(...)` returns a native Python dict; index it    directly with                          │
│ `result['magnetization_mean'\]` or `result['energy_mean'\]`    (no `json.loads` needed -- a local adapter       │
│ handles that for you).                                                                                          │
│  - For T < T_c on an infinite lattice, Onsager gives    |M|(T) = (1 - sinh(2/T)^(-4))^(1/8). You can evaluate   │
│ this with    `calculator_tool`.                                                                                 │
│  - Your `final_answer(...)` must be a single self-contained plain-English    string that restates (T,           │
│ magnetization, energy) for ALL THREE    temperatures and says explicitly how each result compares to theory.    │
│ The reviewer reads ONLY the string you pass to `final_answer`;    stdout prints are invisible to them. A bare   │
│ number or a one-sentence    qualitative remark is NOT an acceptable answer for this three-part    task.         │
│                                                                                                                 │
╰─ LiteLLMModel - ollama_chat/qwen2.5:7b ─────────────────────────────────────────────────────────────────────────╯

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 1 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  results = search_papers(query="2D Ising model simulation parameters")                                            
  print(results)                                                                                                   
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Execution logs:
[1] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.1):
for J x,Jy<0, the material becomes antiferromagnetic. When J = 0, the spins do not interact with each other. In 
these simulations, Jis set as the unit of energy and temperature. The changes in the value of J is explained ahead 
for different anisotropic scenarios.arXiv:2411.06819v1 [cond-mat.stat-mech] 11 Nov 2024 2 By the term anisotropic 
Ising model, we mean thatJy Jx̸= 1.0. This is the model we are interested in our simulations. . FIG. 1. Schematic 
diagram of 2D Ising model with JxandJyas near...

[2] Monte Carlo Simulation of Anisotropic Ising Model Using Metropolis and Wolff Algorithm (arXiv:2411.06819, p.5):
fields in section V. Thereafter, we are presenting the hysteresis loop survey for the Ising model for different 
Isotropic cases in section VI. Finally, we go to the magnetocaloric effect in section VII. And the conclusion of 
the paper in the last section VIII. II. THE METROPOLIS AND WOLFF ALGORITHM The Metropolis and the Wolff algorithm 
have been well studied in the literature for example in [36–40]. Here we are giving the overview of the 
step-by-step process of how these algorithms work [21, 32...

[3] Static and dynamic critical behaviour of 3d random site Ising model: different Monte Carlo algorithms 
(arXiv:cond-mat_0508719, p.1): arXiv:cond-mat/0508719 30 Aug 2005 1 Static and dynamic critical behaviour of 3d 
random site Ising model: different Monte Carlo algorithms D. Ivaneyko a, J. Ilnytskyi b, B. Berche c,d , and Yu. 
Holovatch b,c,a a Ivan Franko National University of Lviv, 79005 Lviv, Ukaine b Institute for Condensed Matter 
Physics, National Academy of Sciences of Ukraine, 79011 Lviv, Ukraine c Laboratoire de Physique des Matériaux, 
Université Henri Poincaré, Nancy 1, 54506 Vandœuvre les Nancy Cedex, France d Instit...

Out: None

...

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ Step 3 ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

─ Executing parsed code: ──────────────────────────────────────────────────────────────────────────────────────── 
  final_answer(result['magnetization_mean'])                                                                       
 ─────────────────────────────────────────────────────────────────────────────────────────────────────────────────

Out - Final answer: 0.20693671875

[Step 3: Duration 8.49 seconds| Input tokens: 11,479 | Output tokens: 239]


--- Actor answer ---
0.20693671875

--- Critic: revise ---
Missing magnetization and energy values at all three temperatures; only one number provided.

================ END-TO-END SUMMARY ================
Accepted on attempt : None
Lessons distilled   : ['Always ensure all required temperatures are covered.', 'Ensure all required measurements are included.', 'Always ensure all required parts of the task are included.']

Final answer:
0.20693671875


## 6. What we saw — and what a single agent can't quite do

This works. A modern AI agent really is five things — an LLM, a reasoning loop, retrieval, external tools, self-correction — and nothing more. You've now built every one.

But look at the trace honestly. Even when the plumbing is clean, four things strain under the weight of being-one-agent:

> **What the run of §5 just looked like.** Three clean attempts, ~3 ReAct Steps each, no crashes — and yet `Accepted on attempt: None`. All three attempts ran `run_ising_simulation` *once* (at a single temperature) and called `final_answer(...)` with a bare float. The Critic correctly caught it each time (*"Missing magnetization and energy at T=2.0 and T=2.27; only one temperature is provided"*) and Reflexion memory filed three on-point lessons (*"Always ensure all required temperatures are covered"*, etc.). So the plumbing is solved: tools invoke cleanly, the reviewer sees the right thing, the memory layer captures the right lesson. What is *not* solved is that a single qwen2.5:7b can't hold "three temperatures × two observables × theory comparison" in working memory long enough to execute it in one ReAct chain, even when the prompt spells it out. That collapse — from a crisp three-part task to a one-number answer — is the evidence for everything below.

- **Tool confusion.** The Actor has *five* tools. On a task that only needs `search_papers` + `run_ising_simulation`, you'll sometimes see it reach for `search_arxiv` (the live web tool) instead — the tool descriptions overlap, and a 7B model is not reliably good at disambiguating under pressure. Two tools would be safer than five. *A team of specialists, each holding two or three tools, would never hit this failure.*

- **Context bloat.** One ReAct chain now carries: the task, a RAG snippet (maybe 500 tokens), three full simulation JSONs (~300 tokens each), intermediate arithmetic, and Critic feedback on revisions. That's thousands of tokens in one window, and small models lose track of *which number came from where*. You will occasionally see the agent cite an energy value as a magnetization, or vice versa. *A team structure gives each agent a scoped context — the Analyst never sees the paper corpus.*

- **Role conflation.** The same model is asked to be the theorist (what does Onsager predict?), the experimentalist (what parameters do I run at?), and the data analyst (does the sim match theory?). One prompt, three personas. qwen2.5:7b does each of the three *individually* fine — but asking it to switch between them inside a single ReAct chain is where coherence drops. *Giving each role its own agent, its own system prompt, and its own tools is the obvious fix — and it's what MAS Part is about.*

- **Critic-as-same-brain.** The Critic is another qwen2.5:7b call. If the Actor is confused about unit conventions on this task, the Critic may be confused about the *same* thing and wave it through. Exercise 1.4.4 (swap in a different model as Critic) is one cheap lever; *a proper Theorist agent reviewing the Experimentalist's output is the structural lever.*

These are not bugs. They are the honest reasons a **team of specialists** tends to beat a **lone generalist** on multi-faceted research tasks. In the MAS Part we build that team.


## 7. Where the MAS Part goes from here

Every SA Part building block is reused in the MAS Part — **nothing is thrown away**.

| SA Part piece | Becomes in the MAS Part |
|---|---|
| `calculator_tool`, `unit_converter_tool` | The Theorist agent's calculation tools |
| `search_papers` (RAG) | The Theorist's literature tool |
| `search_arxiv` (MCP) | The Experimentalist's literature tool |
| `run_ising_simulation` (MCP) | The Experimentalist's primary tool |
| MCP server (`physics_tools_server.py`) | Unchanged — serves both agents |
| Reflection mechanism | Becomes the Theorist's "Review" role |
| Reflexion memory (JSON) | Becomes the team's shared **Lab Notebook** |
| `CodeAgent` | Wrapped as an A2A server for cross-system tasks |

The jump from one agent to a team is structural, not technical. The underlying model and tools stay the same; what changes is that we stop asking one agent to be everything.

**Next up — Module 2.0:** *Why multi-agent?* We revisit today's single agent on a deliberately harder task — a 27-simulation systematic study with scaling collapse — and watch it fail in a way that reflection can't fix. That failure is the door into the MAS Part.


## 8. Exercises

Two only — by design. Module 1.5 is an assembly module, not a source of new ideas.

1. **Turn reflection off.** Set `max_retries=0` and re-run §5. You will see the Actor's raw, unchecked answer. Compare: what does the Critic normally catch that the bare Actor lets through? On this particular task, is the Critic's contribution *physics* or *discipline* (format, self-containment, restated numbers)?

2. **Your own research question.** Replace `TASK_HEADLINE` with a question from your own field. Keep the "practical notes" block (JSON-loads hint, self-contained final_answer, etc.) — those are domain-neutral guardrails. Watch where the failure mode shifts when the task stops being "2D Ising" and starts being something the model knows less about. This is the honest test of whether the agent you've built is *general* or just *Ising-shaped*.


## 9. Clean shutdown

Close the MCP connection and print the Reflexion memory as a parting artefact — "this is what your agent learned in the SA Part."


In [ ]:
_mcp_cm.__exit__(None, None, None)
print("MCP server stopped.")
print("\nReflexion memory at end of Day 1:")
for i, l in enumerate(memory.lessons(), 1):
    print(f"  {i}. {l}")
if not memory.lessons():
    print("  (empty -- the Critic accepted every attempt without correction)")
